In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/hlsimon/chinese-pinyin/pinyin.txt


# Load Dataset

In [2]:
from datasets import load_dataset

ds = load_dataset("Duyu/Pinyin-Hanzi")

README.md:   0%|          | 0.00/223 [00:00<?, ?B/s]

pinyin2hanzi.csv:   0%|          | 0.00/78.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1646553 [00:00<?, ? examples/s]

In [3]:
text_column = '我们试试看！'
pinyin_column = 'wo3 men shi4 shi4 kan4 ！'

dataset = ds['train'].rename_columns({
    text_column: 'text',
    pinyin_column: 'pinyin'
})
dataset

Dataset({
    features: ['text', 'pinyin'],
    num_rows: 1646553
})

In [4]:
for x in dataset:
    print(x)
    break

{'text': '末広雅里', 'pinyin': 'mo4 guang3 ya3 li3'}


## Load Auxiliary Dataset

In [5]:
pinyin_tokens = set()

with open("/kaggle/input/datasets/hlsimon/chinese-pinyin/pinyin.txt", "r") as pinyin_dataset:
    pinyin_data = pinyin_dataset.read()

    for token in pinyin_data.split():
        pinyin_tokens.add(token)

In [6]:
'ma' in pinyin_tokens

True

In [7]:
'cat' in pinyin_tokens

False

**1. Build Vocabulary**

In [8]:
# Build character and pinyin vocabularies from the dataset

from tqdm import tqdm

def build_vocab(dataset):
    """
    dataset: list of dicts with keys 'text' (str) and 'pinyin' (str)
    returns:
        char2idx, idx2char, pinyin2idx, idx2pinyin
    """
    # Collect all characters and pinyin tokens
    chars = set()
    pinyins = set()
    
    for sample in tqdm(dataset):
        text = sample['text']
        pinyin_str = sample['pinyin']
        
        # Add each character from text
        for ch in text:
            chars.add(ch)
        
        # Split pinyin by space and add each token
        tokens = pinyin_str.split()
        for tok in tokens:
            if (
                (tok[-1].isdigit() and tok[:-1] in pinyin_tokens) or \
                (tok[-1].isalpha() and tok in pinyin_tokens)
            ):
                pinyins.add(tok)
    
    # Convert to sorted lists for deterministic order (optional)
    char_list = ['<UNK>', '<PAD>'] + sorted(chars)
    pinyin_list = ['<UNK>', '<PAD>'] + sorted(pinyins)
    
    # Create mappings
    char2idx = {ch: idx for idx, ch in enumerate(char_list)}
    idx2char = {idx: ch for idx, ch in enumerate(char_list)}
    
    pinyin2idx = {p: idx for idx, p in enumerate(pinyin_list)}
    idx2pinyin = {idx: p for idx, p in enumerate(pinyin_list)}
    
    print(f"Number of unique characters (including specials): {len(char_list)}")
    print(f"Number of unique pinyin tokens (including specials): {len(pinyin_list)}")

    return char2idx, idx2char, pinyin2idx, idx2pinyin

In [9]:
char2idx, idx2char, pinyin2idx, idx2pinyin = build_vocab(dataset)

100%|██████████| 1646553/1646553 [00:48<00:00, 33854.94it/s]

Number of unique characters (including specials): 21066
Number of unique pinyin tokens (including specials): 1491


**2. Preprocess Dataset with Mappings**

In [10]:
# Convert text and pinyin to index sequences, filtering invalid pinyin

def preprocess_dataset_filtered(dataset, char2idx, pinyin2idx):
    skipped = 0
    total = 0
    
    for sample in tqdm(dataset):
        total += 1
        text = sample['text']
        pinyin_str = sample['pinyin']
        
        chars = list(text)
        pinyin_tokens = pinyin_str.split()
        
        # Skip if lengths don't match
        if len(chars) != len(pinyin_tokens):
            skipped += 1
            continue
        
        # Skip if any invalid pinyin token
        if any(tok not in pinyin2idx for tok in pinyin_tokens):
            skipped += 1
            continue
        
        # Valid sample - yield the indices
        char_idxs = [char2idx.get(ch, 0) for ch in chars]
        pinyin_idxs = [pinyin2idx[tok] for tok in pinyin_tokens]
        
        yield char_idxs, pinyin_idxs
    
    print(f"Processed {total} samples, skipped {skipped} with invalid pinyin")

In [11]:
preprocessed_dataset = preprocess_dataset_filtered(dataset, char2idx, pinyin2idx)

In [12]:
text_seqs = []
pinyin_seqs = []

for char_idxs, pinyin_idxs in preprocessed_dataset:
    text_seqs.append(char_idxs)
    pinyin_seqs.append(pinyin_idxs)

100%|██████████| 1646553/1646553 [00:56<00:00, 29191.30it/s]

Processed 1646553 samples, skipped 117692 with invalid pinyin


In [13]:
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torch

class PinyinDataset(Dataset):
    """Dataset for character-to-pinyin sequence labeling."""
    def __init__(self, text_sequences, pinyin_sequences):
        """
        Args:
            text_sequences: list of lists of character indices
            pinyin_sequences: list of lists of pinyin indices (aligned with text)
        """
        assert len(text_sequences) == len(pinyin_sequences)
        self.text_seqs = text_sequences
        self.pinyin_seqs = pinyin_sequences

    def __len__(self):
        return len(self.text_seqs)

    def __getitem__(self, idx):
        return {
            'text': torch.tensor(self.text_seqs[idx], dtype=torch.long),
            'pinyin': torch.tensor(self.pinyin_seqs[idx], dtype=torch.long)
        }

def collate_fn(batch, pad_idx=0):
    """
    Collate function to pad sequences within a batch.
    Args:
        batch: list of dicts with 'text' and 'pinyin' tensors
        pad_idx: index of the padding token
    Returns:
        padded_text: (batch_size, max_len) tensor
        padded_pinyin: (batch_size, max_len) tensor
        lengths: (batch_size,) tensor of original lengths (for masking if needed)
    """
    texts = [item['text'] for item in batch]
    pinyins = [item['pinyin'] for item in batch]
    
    # Pad sequences
    padded_text = nn.utils.rnn.pad_sequence(texts, batch_first=True, padding_value=pad_idx)
    padded_pinyin = nn.utils.rnn.pad_sequence(pinyins, batch_first=True, padding_value=pad_idx)
    
    # Optional: return lengths for masking in loss function
    lengths = torch.tensor([len(t) for t in texts], dtype=torch.long)
    
    return padded_text, padded_pinyin, lengths

def create_dataloaders(text_sequences, pinyin_sequences, batch_size, 
                       val_split=0.1, test_split=0.1, pad_idx=0, shuffle=True):
    """
    Split data into train/val/test and create DataLoaders.

    Args:
        text_sequences: list of character index sequences
        pinyin_sequences: list of pinyin index sequences
        batch_size: batch size
        val_split: fraction of data to use for validation (0.0 to 1.0)
        test_split: fraction of data to use for test (0.0 to 1.0)
        pad_idx: padding token index
        shuffle: whether to shuffle training data

    Returns:
        train_loader, val_loader, test_loader
    """
    dataset = PinyinDataset(text_sequences, pinyin_sequences)
    total = len(dataset)
    val_size = int(total * val_split)
    test_size = int(total * test_split)
    train_size = total - val_size - test_size

    assert train_size > 0, "Not enough data for training split"
    assert val_size >= 0 and test_size >= 0, "Split sizes must be non‑negative"

    lengths = [train_size, val_size, test_size]
    train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
        dataset, lengths,
        generator=torch.Generator().manual_seed(42)  # for reproducibility
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=lambda b: collate_fn(b, pad_idx)
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=lambda b: collate_fn(b, pad_idx)
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=lambda b: collate_fn(b, pad_idx)
    )

    return train_loader, val_loader, test_loader

In [14]:
train_loader, val_loader, test_loader = create_dataloaders(
    text_seqs, pinyin_seqs, 
    batch_size=32, 
    val_split=0.1, 
    test_split=0.1,
    pad_idx=1
)

# Define BiLSTM Seq2Seq Model

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BiLSTMForPinyin(nn.Module):
    """
    BiLSTM model for predicting pinyin tokens from character sequences.
    For each character in the input, it outputs a probability distribution over pinyin tokens.
    """
    def __init__(self, vocab_size_char, vocab_size_pinyin, embedding_dim, hidden_dim, num_layers=2, dropout=0.3, pad_idx=1):
        """
        Args:
            vocab_size_char: number of unique characters (including <PAD> and <UNK>)
            vocab_size_pinyin: number of unique pinyin tokens (including <PAD> and <UNK>)
            embedding_dim: dimension of character embeddings
            hidden_dim: dimension of LSTM hidden states (will be doubled due to bidirectionality)
            num_layers: number of LSTM layers
            dropout: dropout probability (applied between LSTM layers and after LSTM)
            pad_idx: index of the padding token (used to ignore padding in loss)
        """
        super().__init__()
        self.embedding = nn.Embedding(vocab_size_char, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        # After BiLSTM, hidden_dim * 2 because bidirectional
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, vocab_size_pinyin)
        self.pad_idx = pad_idx

    def forward(self, x):
        """
        Args:
            x: (batch_size, seq_len) tensor of character indices
        Returns:
            logits: (batch_size, seq_len, vocab_size_pinyin)
        """
        # Embedding: (batch_size, seq_len, embedding_dim)
        emb = self.embedding(x)
        
        # LSTM: outputs (batch_size, seq_len, hidden_dim * 2)
        lstm_out, _ = self.lstm(emb)
        
        # Apply dropout
        lstm_out = self.dropout(lstm_out)
        
        # Linear layer to produce logits for each pinyin token
        logits = self.fc(lstm_out)  # (batch_size, seq_len, vocab_size_pinyin)
        
        return logits

    def predict(self, x):
        """Returns predicted pinyin indices (no grad)"""
        self.eval()
        with torch.no_grad():
            logits = self.forward(x)
            preds = torch.argmax(logits, dim=-1)  # (batch_size, seq_len)
        return preds

# Train Model

In [16]:
# Training loop with tqdm, loss tracking, and validation accuracy

import torch
import torch.nn as nn
from tqdm import tqdm
import numpy as np

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Hyperparameters
vocab_size_char = len(char2idx)
vocab_size_pinyin = len(pinyin2idx)
embedding_dim = 100
hidden_dim = 256
num_layers = 2
dropout = 0.3
pad_idx = 1  # <PAD> token index

Using device: cuda


In [17]:
# Instantiate model
model = BiLSTMForPinyin(
    vocab_size_char=vocab_size_char,
    vocab_size_pinyin=vocab_size_pinyin,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    dropout=dropout,
    pad_idx=pad_idx
)
model.to(device)

BiLSTMForPinyin(
  (embedding): Embedding(21066, 100, padding_idx=1)
  (lstm): LSTM(100, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=512, out_features=1491, bias=True)
)

In [18]:
# Loss function (ignore padding)
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)

In [19]:
# Training settings
num_epochs = 1
patience = 3
counter = 0
best_val_loss = float('inf')

for epoch in range(num_epochs):
    # --- Training ---
    model.train()
    total_train_loss = 0
    train_steps = 0
    
    # Use tqdm for training batches
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]')
    for batch_text, batch_pinyin, lengths in train_pbar:
        batch_text = batch_text.to(device)
        batch_pinyin = batch_pinyin.to(device)
        
        # Forward pass
        logits = model(batch_text)  # (batch, seq_len, vocab_pinyin)
        
        # Reshape for loss: (batch*seq_len, vocab_pinyin) vs (batch*seq_len)
        loss = criterion(logits.reshape(-1, vocab_size_pinyin), batch_pinyin.reshape(-1))
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        train_steps += 1
        
        # Update tqdm description with current loss
        train_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_train_loss = total_train_loss / train_steps
    
    # --- Validation ---
    model.eval()
    total_val_loss = 0
    val_steps = 0
    correct_predictions = 0
    total_predictions = 0
    
    val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Val]')
    with torch.no_grad():
        for batch_text, batch_pinyin, lengths in val_pbar:
            batch_text = batch_text.to(device)
            batch_pinyin = batch_pinyin.to(device)
            
            logits = model(batch_text)  # (batch, seq_len, vocab_pinyin)
            
            # Loss (same as training)
            loss = criterion(logits.reshape(-1, vocab_size_pinyin), batch_pinyin.reshape(-1))
            total_val_loss += loss.item()
            val_steps += 1
            
            # Accuracy (ignore padding)
            predictions = logits.argmax(dim=-1)  # (batch, seq_len)
            
            # Create mask for non-padded positions
            mask = batch_pinyin != pad_idx
            correct = (predictions == batch_pinyin) & mask
            correct_predictions += correct.sum().item()
            total_predictions += mask.sum().item()
            
            val_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_val_loss = total_val_loss / val_steps
    val_accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    
    # Print epoch summary
    print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}, Val Acc = {val_accuracy:.4f}")
    
    # --- Early Stopping Logic ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'text2pinyin_best_model.pt')
        print(f"  -> Best model saved (val loss: {best_val_loss:.4f})")
        counter = 0
    else:
        counter += 1
        print(f"  -> No improvement for {counter} epoch(s).")
        if counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break

print("Training complete!")

Epoch 1/1 [Val]: 100%|██████████| 4778/4778 [00:20<00:00, 235.80it/s, loss=0.0967]


Epoch 1: Train Loss = 0.4683, Val Loss = 0.0934, Val Acc = 0.9865
  -> Best model saved (val loss: 0.0934)
Training complete!


## Load the Best Model for Testing

In [20]:
import torch
import torch.nn as nn

vocab_size_char = len(char2idx)
vocab_size_pinyin = len(pinyin2idx)
embedding_dim = 100
hidden_dim = 256
num_layers = 2
dropout = 0.3
pad_idx = 1  # <PAD> token index

In [21]:
text2pinyin = BiLSTMForPinyin(
    vocab_size_char=vocab_size_char,
    vocab_size_pinyin=vocab_size_pinyin,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    dropout=dropout,
    pad_idx=pad_idx
)

# Load the best saved weights
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
text2pinyin.load_state_dict(torch.load('text2pinyin_best_model.pt', map_location=device))
text2pinyin.to(device)
text2pinyin.eval()

BiLSTMForPinyin(
  (embedding): Embedding(21066, 100, padding_idx=1)
  (lstm): LSTM(100, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=512, out_features=1491, bias=True)
)

In [22]:
def predict_pinyin(sentence, model, char2idx, idx2pinyin, device):
    """
    Predict pinyin sequence for a single Chinese sentence.
    Args:
        sentence: string of Chinese characters (no spaces)
        model: trained BiLSTMForPinyin model
        char2idx: char to index mapping
        idx2pinyin: index to pinyin token mapping
        device: torch device
    Returns:
        pinyin_tokens: list of predicted pinyin strings
    """
    # Tokenize into characters
    chars = list(sentence)
    # Convert to indices, use 0 for unknown characters
    indices = [char2idx.get(ch, 0) for ch in chars]
    # Create tensor with batch dimension
    input_tensor = torch.tensor([indices], dtype=torch.long).to(device)
    
    with torch.no_grad():
        logits = model(input_tensor)           # (1, seq_len, vocab_pinyin)
        pred_indices = logits.argmax(dim=-1)   # (1, seq_len)
    
    # Convert indices to pinyin tokens
    pred_tokens = [idx2pinyin[idx.item()] for idx in pred_indices[0]]
    return pred_tokens

In [23]:
# Test sentences (example Chinese sentences)
test_sentences = [
    "你好世界",
    "今天天气不错",
    "我喜欢吃苹果",
    "机器学习很有趣",
    "末広雅里"   # from the dataset example
]

print("Testing model on example sentences:\n")
for sentence in test_sentences:
    predicted = predict_pinyin(sentence, text2pinyin, char2idx, idx2pinyin, device)
    print(f"Input  : {sentence}")
    print(f"Output : {' '.join(predicted)}")
    print()

Testing model on example sentences:

Input  : 你好世界
Output : ni3 hao3 shi4 jie4

Input  : 今天天气不错
Output : jin1 tian1 tian1 qi4 bu4 cuo4

Input  : 我喜欢吃苹果
Output : wo3 xi3 huan1 chi1 ping2 guo3

Input  : 机器学习很有趣
Output : ji1 qi4 xue2 xi2 hen3 you3 qu4

Input  : 末広雅里
Output : mo4 guang3 ya3 li3



# Define Inverse Objective - Pinyin to Characters

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BiLSTMForChar(nn.Module):
    """
    BiLSTM model for predicting character tokens from pinyin sequences.
    For each character in the input, it outputs a probability distribution over character tokens.
    """
    def __init__(self, vocab_size_char, vocab_size_pinyin, embedding_dim, hidden_dim, num_layers=2, dropout=0.3, pad_idx=1):
        """
        Args:
            vocab_size_char: number of unique characters (including <PAD> and <UNK>)
            vocab_size_pinyin: number of unique pinyin tokens (including <PAD> and <UNK>)
            embedding_dim: dimension of character embeddings
            hidden_dim: dimension of LSTM hidden states (will be doubled due to bidirectionality)
            num_layers: number of LSTM layers
            dropout: dropout probability (applied between LSTM layers and after LSTM)
            pad_idx: index of the padding token (used to ignore padding in loss)
        """
        super().__init__()
        self.embedding = nn.Embedding(vocab_size_pinyin, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        # After BiLSTM, hidden_dim * 2 because bidirectional
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, vocab_size_char)
        self.pad_idx = pad_idx

    def forward(self, x):
        """
        Args:
            x: (batch_size, seq_len) tensor of character indices
        Returns:
            logits: (batch_size, seq_len, vocab_size_pinyin)
        """
        # Embedding: (batch_size, seq_len, embedding_dim)
        emb = self.embedding(x)
        
        # LSTM: outputs (batch_size, seq_len, hidden_dim * 2)
        lstm_out, _ = self.lstm(emb)
        
        # Apply dropout
        lstm_out = self.dropout(lstm_out)
        
        # Linear layer to produce logits for each pinyin token
        logits = self.fc(lstm_out)  # (batch_size, seq_len, vocab_size_pinyin)
        
        return logits

    def predict(self, x):
        """Returns predicted pinyin indices (no grad)"""
        self.eval()
        with torch.no_grad():
            logits = self.forward(x)
            preds = torch.argmax(logits, dim=-1)  # (batch_size, seq_len)
        return preds

In [25]:
# Training loop with tqdm, loss tracking, and validation accuracy

import torch
import torch.nn as nn
from tqdm import tqdm
import numpy as np

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Hyperparameters
vocab_size_char = len(char2idx)
vocab_size_pinyin = len(pinyin2idx)
embedding_dim = 100
hidden_dim = 256
num_layers = 2
dropout = 0.3
pad_idx = 1  # <PAD> token index

Using device: cuda


In [26]:
# Instantiate model
model = BiLSTMForChar(
    vocab_size_char=vocab_size_char,
    vocab_size_pinyin=vocab_size_pinyin,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    dropout=dropout,
    pad_idx=pad_idx
)
model.to(device)

BiLSTMForChar(
  (embedding): Embedding(1491, 100, padding_idx=1)
  (lstm): LSTM(100, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=512, out_features=21066, bias=True)
)

In [27]:
# Loss function (ignore padding)
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

In [28]:
# Training settings
num_epochs = 20
patience = 3
counter = 0
best_val_loss = float('inf')

for epoch in range(num_epochs):
    # --- Training ---
    model.train()
    total_train_loss = 0
    train_steps = 0
    
    # Use tqdm for training batches
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]')
    for batch_text, batch_pinyin, lengths in train_pbar:
        batch_text = batch_text.to(device)
        batch_pinyin = batch_pinyin.to(device)
        
        # Forward pass
        logits = model(batch_pinyin)  # (batch, seq_len, vocab_char)
        
        # Reshape for loss: (batch*seq_len, vocab_char) vs (batch*seq_len)
        loss = criterion(logits.reshape(-1, vocab_size_char), batch_text.reshape(-1))
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        train_steps += 1
        
        # Update tqdm description with current loss
        train_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_train_loss = total_train_loss / train_steps
    
    # --- Validation ---
    model.eval()
    total_val_loss = 0
    val_steps = 0
    correct_predictions = 0
    total_predictions = 0
    
    val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Val]')
    with torch.no_grad():
        for batch_text, batch_pinyin, lengths in val_pbar:
            batch_text = batch_text.to(device)
            batch_pinyin = batch_pinyin.to(device)
            
            logits = model(batch_pinyin)  # (batch, seq_len, vocab_char)
            
            # Loss (same as training)
            loss = criterion(logits.reshape(-1, vocab_size_char), batch_text.reshape(-1))
            total_val_loss += loss.item()
            val_steps += 1
            
            # Accuracy (ignore padding)
            predictions = logits.argmax(dim=-1)  # (batch, seq_len)
            
            # Create mask for non-padded positions
            mask = batch_text != pad_idx
            correct = (predictions == batch_text) & mask
            correct_predictions += correct.sum().item()
            total_predictions += mask.sum().item()
            
            val_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_val_loss = total_val_loss / val_steps
    val_accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    
    # Print epoch summary
    print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}, Val Acc = {val_accuracy:.4f}")
    
    # --- Early Stopping Logic ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'pinyin2text_best_model.pt')
        print(f"  -> Best model saved (val loss: {best_val_loss:.4f})")
        counter = 0
    else:
        counter += 1
        print(f"  -> No improvement for {counter} epoch(s).")
        if counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break

print("Training complete!")

Epoch 1/20 [Val]: 100%|██████████| 4778/4778 [00:27<00:00, 173.36it/s, loss=0.6423]


Epoch 1: Train Loss = 0.6258, Val Loss = 0.4604, Val Acc = 0.8814
  -> Best model saved (val loss: 0.4604)


Epoch 2/20 [Val]: 100%|██████████| 4778/4778 [00:29<00:00, 163.40it/s, loss=0.5904]


Epoch 2: Train Loss = 0.4917, Val Loss = 0.4222, Val Acc = 0.8901
  -> Best model saved (val loss: 0.4222)


Epoch 3/20 [Val]: 100%|██████████| 4778/4778 [00:26<00:00, 180.70it/s, loss=0.5040]


...

Epoch 18: Train Loss = 0.3292, Val Loss = 0.3307, Val Acc = 0.9170
  -> Best model saved (val loss: 0.3307)


Epoch 19/20 [Val]: 100%|██████████| 4778/4778 [00:27<00:00, 174.57it/s, loss=0.3606]


Epoch 19: Train Loss = 0.3270, Val Loss = 0.3317, Val Acc = 0.9176
  -> No improvement for 1 epoch(s).


Epoch 20/20 [Val]: 100%|██████████| 4778/4778 [00:27<00:00, 171.65it/s, loss=0.3387]

Epoch 20: Train Loss = 0.3247, Val Loss = 0.3338, Val Acc = 0.9179
  -> No improvement for 2 epoch(s).
Training complete!


## Load the Best Model for Testing

In [29]:
import torch
import torch.nn as nn

vocab_size_char = len(char2idx)
vocab_size_pinyin = len(pinyin2idx)
embedding_dim = 100
hidden_dim = 256
num_layers = 2
dropout = 0.3
pad_idx = 1  # <PAD> token index

In [30]:
pinyin2text = BiLSTMForChar(
    vocab_size_char=vocab_size_char,
    vocab_size_pinyin=vocab_size_pinyin,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    dropout=dropout,
    pad_idx=pad_idx
)

# Load the best saved weights
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pinyin2text.load_state_dict(torch.load('pinyin2text_best_model.pt', map_location=device))
pinyin2text.to(device)
pinyin2text.eval()

BiLSTMForChar(
  (embedding): Embedding(1491, 100, padding_idx=1)
  (lstm): LSTM(100, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=512, out_features=21066, bias=True)
)

In [31]:
def test_model(model, test_loader, pad_idx, vocab_size_char, device):
    """
    Evaluate the pinyin2text model on the test set.

    Args:
        model: trained model
        test_loader: DataLoader for test set (returns batch_text, batch_pinyin, lengths)
        pad_idx: index used for padding (to ignore in accuracy)
        vocab_size_char: size of character vocabulary
        device: torch device

    Returns:
        test_accuracy: float (0–1)
    """
    model.eval()
    correct_predictions = 0
    total_predictions = 0
    num_batches = 0

    with torch.no_grad():
        for batch_text, batch_pinyin, lengths in tqdm(test_loader, desc='Testing'):
            batch_text = batch_text.to(device)
            batch_pinyin = batch_pinyin.to(device)

            logits = model(batch_pinyin)  # (batch, seq_len, vocab_char)

            num_batches += 1

            # Accuracy (ignore padding)
            predictions = logits.argmax(dim=-1)
            mask = batch_text != pad_idx
            correct = (predictions == batch_text) & mask
            correct_predictions += correct.sum().item()
            total_predictions += mask.sum().item()

    test_accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0

    return test_accuracy

In [32]:
test_acc = test_model(pinyin2text, test_loader, pad_idx, vocab_size_char, device)
print(f"Test Accuracy: {test_acc:.4f}")

Testing: 100%|██████████| 4778/4778 [00:19<00:00, 242.52it/s]

Test Accuracy: 0.9170


In [33]:
def predict_char(pinyin_sequence, model, pinyin2idx, idx2char, device):
    """
    Predict char sequence for a single Chinese pinyin sequence.
    Args:
        sentence: string of Chinese characters (no spaces)
        model: trained BiLSTMForChar model
        pinyin2idx: pinyin to index mapping
        idx2char: index to char token mapping
        device: torch device
    Returns:
        char_tokens: list of predicted char strings
    """
    # Tokenize into characters
    pinyin_tokens = pinyin_sequence.split()
    # Convert to indices, use 0 for unknown characters
    indices = [pinyin2idx.get(pinyin_token, 0) for pinyin_token in pinyin_tokens]
    # Create tensor with batch dimension
    input_tensor = torch.tensor([indices], dtype=torch.long).to(device)
    
    with torch.no_grad():
        logits = model(input_tensor)           # (1, seq_len, vocab_pinyin)
        pred_indices = logits.argmax(dim=-1)   # (1, seq_len)
    
    # Convert indices to pinyin tokens
    pred_tokens = [idx2char[idx.item()] for idx in pred_indices[0]]
    return pred_tokens

In [34]:
# Test sentences (example Chinese sentences)
test_sentences = [
    "ni2 hao3 shi4 jie4",
    "jin1 tian1 tian1 qi4 bu4 cuo4",
    "wo3 xi3 huan1 chi1 ping2 guo3",
    "ji1 qi4 xue2 xi2 hen3 you3 qu4",
    "mo4 guang3 ya3 li3",
]

print("Testing model on example sentences:\n")
for sentence in test_sentences:
    predicted = predict_char(sentence, pinyin2text, pinyin2idx, idx2char, device)
    print(f"Input  : {sentence}")
    print(f"Output : {' '.join(predicted)}")
    print()

Testing model on example sentences:

Input  : ni2 hao3 shi4 jie4
Output : 泥 好 世 界

Input  : jin1 tian1 tian1 qi4 bu4 cuo4
Output : 今 天 天 气 不 错

Input  : wo3 xi3 huan1 chi1 ping2 guo3
Output : 我 喜 欢 吃 苹 果

Input  : ji1 qi4 xue2 xi2 hen3 you3 qu4
Output : 机 器 学 习 很 有 趣

Input  : mo4 guang3 ya3 li3
Output : 莫 广 雅 里

